In [1]:
# IMPORTS
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from xgboost import XGBClassifier
from scipy.stats import rankdata
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('✓ Imports complete')


✓ Imports complete


In [2]:
# CONFIGURATION
SEED = 42
NSPLITS = 5

# Load data
path = Path('/kaggle/input/playground-series-s6e2')
train = pd.read_csv(path / 'train.csv', index_col=0)
test = pd.read_csv(path / 'test.csv', index_col=0)

train.columns = [c.strip().lower().replace(' ', '_') for c in train.columns]
test.columns = [c.strip().lower().replace(' ', '_') for c in test.columns]

ystr = train.columns[-1]
X = train.drop(columns=ystr).copy()
y = (train[ystr] == 'Presence').astype(int)
X_test = test.copy()

print(f'Train: {X.shape}, Test: {X_test.shape}')
print(f'Target dist: 0={sum(y==0)}, 1={sum(y==1)}')


Train: (630000, 13), Test: (270000, 13)
Target dist: 0=347546, 1=282454


In [3]:
# LABEL ENCODE CATEGORICALS
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])
    le_dict[col] = le

print(f'✓ Encoded {len(cat_cols)} categorical columns')


✓ Encoded 0 categorical columns


In [4]:
# FAST FEATURE ENGINEERING
def fe_fast(X_tr, X_val, y_tr):
    """Minimal, fast feature engineering"""
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    
    # Binary features (domain knowledge)
    if 'age' in X_tr.columns:
        X_tr['age_gt55'] = (X_tr['age'] > 55).astype(int)
        X_val['age_gt55'] = (X_val['age'] > 55).astype(int)
        X_tr['age_gt65'] = (X_tr['age'] > 65).astype(int)
        X_val['age_gt65'] = (X_val['age'] > 65).astype(int)
    
    if 'bp' in X_tr.columns:
        X_tr['bp_high'] = (X_tr['bp'] > 140).astype(int)
        X_val['bp_high'] = (X_val['bp'] > 140).astype(int)
    
    if 'cholesterol' in X_tr.columns:
        X_tr['chol_high'] = (X_tr['cholesterol'] > 240).astype(int)
        X_val['chol_high'] = (X_val['cholesterol'] > 240).astype(int)
    
    # Polynomial features (top 2 only)
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()[:2]
    for col in num_cols:
        X_tr[f'{col}_sq'] = X_tr[col] ** 2
        X_val[f'{col}_sq'] = X_val[col] ** 2
    
    # Target encoding (fast)
    base_features = [c for c in X_tr.columns if c not in X_tr.select_dtypes(include=['int']).columns or X_tr[c].nunique() < 20]
    
    global_mean = y_tr.mean()
    temp = pd.concat([X_tr, y_tr], axis=1)
    
    for col in base_features[:5]:  # Only top 5
        try:
            agg_dict = temp.groupby(col)[y_tr.name].agg(['mean', 'count'])
            agg_dict['mean'] = (agg_dict['count'] * agg_dict['mean'] + 10 * global_mean) / (agg_dict['count'] + 10)
            X_tr[f'te_{col}'] = X_tr[col].map(agg_dict['mean']).fillna(global_mean)
            X_val[f'te_{col}'] = X_val[col].map(agg_dict['mean']).fillna(global_mean)
        except:
            pass
    
    # Fill NaN
    X_tr = X_tr.fillna(0)
    X_val = X_val.fillna(0)
    
    return X_tr.astype('float32'), X_val.astype('float32')


def fe_test_fast(X_test, X_train, y_train):
    """Same FE for test"""
    X_test = X_test.copy()
    
    if 'age' in X_test.columns:
        X_test['age_gt55'] = (X_test['age'] > 55).astype(int)
        X_test['age_gt65'] = (X_test['age'] > 65).astype(int)
    if 'bp' in X_test.columns:
        X_test['bp_high'] = (X_test['bp'] > 140).astype(int)
    if 'cholesterol' in X_test.columns:
        X_test['chol_high'] = (X_test['cholesterol'] > 240).astype(int)
    
    num_cols = X_test.select_dtypes(include=[np.number]).columns.tolist()[:2]
    for col in num_cols:
        X_test[f'{col}_sq'] = X_test[col] ** 2
    
    global_mean = y_train.mean()
    temp = pd.concat([X_train, y_train], axis=1)
    
    base_features = [c for c in X_train.columns if c not in X_train.select_dtypes(include=['int']).columns or X_train[c].nunique() < 20]
    
    for col in base_features[:5]:
        try:
            agg_dict = temp.groupby(col)[y_train.name].agg(['mean', 'count'])
            agg_dict['mean'] = (agg_dict['count'] * agg_dict['mean'] + 10 * global_mean) / (agg_dict['count'] + 10)
            X_test[f'te_{col}'] = X_test[col].map(agg_dict['mean']).fillna(global_mean)
        except:
            pass
    
    return X_test.fillna(0).astype('float32')

print('✓ Feature engineering functions ready')


✓ Feature engineering functions ready


In [5]:
# OPTIMIZED HYPERPARAMETERS (from Optuna or manual tuning)
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': SEED,
    'n_jobs': -1,
    'learning_rate': 0.025,
    'max_depth': 3,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'gamma': 0.5,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'min_child_weight': 2,
    'n_estimators': 12000,
    'early_stopping_rounds': 200,
}

# Variants for diversity
xgb_params_v2 = xgb_params.copy()
xgb_params_v2['learning_rate'] = 0.03
xgb_params_v2['max_depth'] = 4
xgb_params_v2['random_state'] = SEED + 1
xgb_params_v2['n_estimators'] = 10000

xgb_params_v3 = xgb_params.copy()
xgb_params_v3['learning_rate'] = 0.02
xgb_params_v3['max_depth'] = 2
xgb_params_v3['random_state'] = SEED + 2
xgb_params_v3['n_estimators'] = 15000

configs = {'v1': xgb_params, 'v2': xgb_params_v2, 'v3': xgb_params_v3}

print('✓ 3 model configurations ready')


✓ 3 model configurations ready


In [6]:
# TRAIN MODELS WITH 5-FOLD CV
print('\n' + '='*70)
print('TRAINING MODELS')
print('='*70)

kf = StratifiedKFold(n_splits=NSPLITS, shuffle=True, random_state=SEED)

all_oof = {}
all_test = {}

for model_name, params in configs.items():
    print(f'\nTraining {model_name}...')
    
    oof_train = np.zeros(len(X))
    oof_test = np.zeros(len(X_test))
    cv_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # FE
        X_tr_fe, X_val_fe = fe_fast(X_tr, X_val, y_tr)
        
        # Train
        model = XGBClassifier(**params)
        model.fit(X_tr_fe, y_tr, eval_set=[(X_val_fe, y_val)], verbose=False)
        
        # Predict
        oof_train[val_idx] = model.predict_proba(X_val_fe)[:, 1]
        
        # Test
        X_test_fe = fe_test_fast(X_test, X_tr, y_tr)
        oof_test += model.predict_proba(X_test_fe)[:, 1] / NSPLITS
        
        auc = roc_auc_score(y_val, oof_train[val_idx])
        cv_scores.append(auc)
        print(f'  Fold {fold+1}: {auc:.6f}')
    
    all_oof[model_name] = oof_train
    all_test[model_name] = oof_test
    
    oof_auc = roc_auc_score(y, oof_train)
    print(f'  OOF AUC: {oof_auc:.6f} (±{np.std(cv_scores):.5f})')

print('\n' + '='*70)



TRAINING MODELS

Training v1...
  Fold 1: 0.955874
  Fold 2: 0.954810
  Fold 3: 0.955625
  Fold 4: 0.955227
  Fold 5: 0.956056
  OOF AUC: 0.955517 (±0.00045)

Training v2...
  Fold 1: 0.955805
  Fold 2: 0.954730
  Fold 3: 0.955549
  Fold 4: 0.955163
  Fold 5: 0.955961
  OOF AUC: 0.955440 (±0.00045)

Training v3...
  Fold 1: 0.955979
  Fold 2: 0.954861
  Fold 3: 0.955696
  Fold 4: 0.955297
  Fold 5: 0.956130
  OOF AUC: 0.955592 (±0.00046)



In [7]:
# ENSEMBLE
print('\nEnsembling...')

# Average
avg_oof = np.mean(list(all_oof.values()), axis=0)
avg_test = np.mean(list(all_test.values()), axis=0)

# Weighted
scores = {k: roc_auc_score(y, v) for k, v in all_oof.items()}
weights = np.array([scores[k]**2 for k in all_oof.keys()])
weights /= weights.sum()

weighted_oof = sum(all_oof[k] * weights[i] for i, k in enumerate(all_oof.keys()))
weighted_test = sum(all_test[k] * weights[i] for i, k in enumerate(all_test.keys()))

# Stacking
stack_X = np.column_stack([all_oof[k] for k in all_oof.keys()])
stack_test = np.column_stack([all_test[k] for k in all_test.keys()])

meta = Ridge(alpha=1.0, random_state=SEED)
meta.fit(stack_X, y)

stack_oof = np.clip(meta.predict(stack_X), 0, 1)
stack_test_pred = np.clip(meta.predict(stack_test), 0, 1)

# Rank
rank_oof = np.column_stack([rankdata(all_oof[k])/len(y) for k in all_oof.keys()]).mean(axis=1)
rank_test = np.column_stack([rankdata(all_test[k])/len(X_test) for k in all_test.keys()]).mean(axis=1)

# Final blend
final_oof = 0.4*stack_oof + 0.3*weighted_oof + 0.2*rank_oof + 0.1*avg_oof
final_test = 0.4*stack_test_pred + 0.3*weighted_test + 0.2*rank_test + 0.1*avg_test

print(f'\nAverage:  {roc_auc_score(y, avg_oof):.6f}')
print(f'Weighted: {roc_auc_score(y, weighted_oof):.6f}')
print(f'Stacked:  {roc_auc_score(y, stack_oof):.6f}')
print(f'Rank:     {roc_auc_score(y, rank_oof):.6f}')
print(f'\n FINAL: {roc_auc_score(y, final_oof):.6f}')



Ensembling...

Average:  0.955552
Weighted: 0.955553
Stacked:  0.955592
Rank:     0.955553

 FINAL: 0.955579


In [8]:
# SUBMISSION
submission = pd.DataFrame({
    'id': test.index,
    'Heart Disease': final_test
})

submission.to_csv('submission.csv', index=False)

print('\n' + '='*70)
print('SUBMISSION SAVED')
print('='*70)
print(submission.head(10))
print(f'\nStats: min={final_test.min():.4f}, max={final_test.max():.4f}, mean={final_test.mean():.4f}')



SUBMISSION SAVED
       id  Heart Disease
0  630000       0.914056
1  630001       0.021994
2  630002       0.972073
3  630003       0.011147
4  630004       0.249847
5  630005       0.961868
6  630006       0.009315
7  630007       0.607721
8  630008       0.978597
9  630009       0.031798

Stats: min=0.0002, max=0.9999, mean=0.4599
